In [ ]:
import torch
import numpy as np
import copy
import gc
from tqdm import tqdm
import matplotlib.pyplot as plt
from torchdiffeq import odeint
import sys
sys.path.append('../../..')
from helpers import objective, adj_lattice, criticalK,adjoint_init, grad_estimate, order_param, adabelief_update, AGM_ODE, ODE_Adjoint
from configs.agm_cfg import cfg
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# --- 1. Configuration & Constants ---
# Using a namespace keeps global variables organized

# --- 2. Data Preparation ---
# Pulling cached data
%store -r adj_matrix
state_size = adj_matrix.shape[1]

# Frequency generation (Optimized)
dist = torch.distributions.Normal(0, 1)
low_cdf = dist.cdf(torch.tensor(-3.0).to(device))
high_cdf = dist.cdf(torch.tensor(3.0).to(device))
u_rand = torch.rand(cfg.batch_size, state_size, device=device)
freqs = dist.icdf(low_cdf + u_rand * (high_cdf - low_cdf)) / cfg.scale
%store freqs

# Critical K Calculation
Kcrit_val, L_mat = criticalK(adj_matrix, state_size, cfg.scale)
K_ctrl = 0.01 * Kcrit_val

# Tensor Factory
ctx = {'device': device, 'dtype': torch.float32}
A_torch = torch.as_tensor(adj_matrix, **ctx)
L_torch = torch.as_tensor(L_mat, **ctx)
ts_exp = torch.linspace(0, cfg.t_total, cfg.t_size).to(device)
dt_val = ts_exp[1] - ts_exp[0]

# Initial Conditions
ys_ran = torch.randn(cfg.batch_size, state_size, **ctx) / cfg.scale_kin

# --- 3. Optimization Setup ---
# Initialize control field (signal to be optimized)
force_field = torch.zeros(cfg.t_size, cfg.batch_size, 1, **ctx)

# Instantiate ODE Systems
SML = AGM_ODE(cfg.batch_size, state_size, K_ctrl, A_torch, freqs, force_field, dt_val).to(device)
SMD = ODE_Adjoint(cfg.batch_size, state_size, K_ctrl, freqs, A_torch, force_field, ys_ran, dt_val, device).to(device)

# Optimizer State (Manual AdaBelief buffers)
opt_state = {
    'm': torch.zeros_like(force_field),
    'v': torch.zeros_like(force_field),
    'v_hat_max': torch.zeros_like(force_field),
    'steps': 0
}

# Metrics storage
history = {"loss": [], "ste": [], "order_p": []}

# --- 4. Optimization Loop ---
pbar = tqdm(range(cfg.iterations), desc="Adjoint Optimization")

for j in pbar:
    # A. Forward Pass
    ys_ini = torch.zeros(cfg.batch_size, 1, **ctx)
    ys_exp = torch.cat((ys_ran, ys_ini), dim=1)
    
    with torch.no_grad():
        # Using Euler for speed as per your original logic
        ys_tray = odeint(SML, ys_exp, ts_exp, method='euler')

    # B. Loss & Metrics Calculation
    # 
    obj = objective(ys_tray[-1, :, 0:state_size], A_torch)
    kin_energy = cfg.R_penalty * ys_tray[-1, :, -1] * state_size
    loss_vec = obj + kin_energy
    
    avg_loss = loss_vec.mean().item()
    ord_param = torch.mean(order_param(ys_tray[-1, :, 0:state_size], L_torch)).item()
    
    # C. Backward Pass (Adjoint Method)
    ys_phases = ys_tray[:, :, 0:state_size]
    initial_adjoint = adjoint_init(ys_phases[-1], state_size)
    
    # Sync states to Adjoint Evaluator
    SMD.x = ys_phases
    SMD.u = force_field
    
    with torch.no_grad():
        p_tray = odeint(SMD, initial_adjoint, ts_exp, method='euler')

    # D. Gradient Estimation & Optimizer Step
    # Flip the adjoint trajectory to match forward time
    p_reversed = torch.flip(p_tray, [0])
    grad = 2 * cfg.R_penalty * force_field + grad_estimate(p_reversed, ys_phases, state_size, K_ctrl)

    with torch.no_grad():
        force_field, opt_state['m'], opt_state['v'], opt_state['v_hat_max'], opt_state['steps'] = adabelief_update(
            force_field, grad, opt_state['m'], opt_state['v'], opt_state['v_hat_max'], opt_state['steps'], lr=cfg.learning_rate
        )
    
    # E. Update SML for next iteration
    SML.u = force_field
    
    # F. Logging
    history["loss"].append(avg_loss)
    history["ste"].append(loss_vec.std().item() / np.sqrt(cfg.batch_size))
    history["order_p"].append(ord_param)

    if j % 10 == 0:
        pbar.set_postfix({"Loss": f"{avg_loss:.4f}", "OrderP": f"{ord_param:.3f}"})

    # G. Memory Management
    if j % 500 == 0:
        gc.collect()
        torch.cuda.empty_cache()

# --- 5. Final Visualization ---
plt.figure(figsize=(10, 5))
t_axis = np.arange(cfg.iterations)
perf = np.array(history["loss"])
ste = np.array(history["ste"])

plt.plot(t_axis, perf, label='Mean Loss')
plt.fill_between(t_axis, perf - ste, perf + ste, alpha=0.3)
plt.title("Adjoint Optimization Performance")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.legend()
plt.grid(True)
plt.show()